# Tacotron 2

A self-contained refresher on **Tacotron 2** — the canonical neural text-to-speech (TTS) pipeline that maps text → mel spectrogram → waveform.

**Domain:** Speech & Audio  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**Tacotron 2** (Shen et al., Google, 2018) is a neural **text-to-speech** system. It maps a sequence of characters (or phonemes) directly to a **mel spectrogram** using a recurrent **seq2seq encoder–decoder with location-sensitive attention**, and then a *separate* neural **vocoder** (WaveNet in the paper; WaveGlow / HiFi-GAN in practice) turns that mel spectrogram into an audio waveform.

**The problem it solves.** Older TTS was either *concatenative* (stitch recorded units — large databases, glitchy joins) or *parametric* (vocode hand-engineered linguistic features — buzzy, unnatural). Tacotron 2 learns the whole acoustic mapping end-to-end from `(text, audio)` pairs, with **near-human naturalness** (the paper reports a MOS of 4.53 vs 4.58 for ground-truth recordings) and **no hand-built front-end**.

**Reach for it when** you want a well-understood, hackable autoregressive TTS baseline, a single-speaker voice with quality over latency, or to *learn* the text→mel→vocoder decomposition that almost every modern TTS still echoes.

**Skip it when** you need real-time / streaming low-latency synthesis (autoregressive decoding is inherently slow), rock-solid robustness on long or out-of-domain text (attention can derail), or zero-shot voice cloning. Non-autoregressive successors — FastSpeech 2, [[vits]], [[styletts2]] — are faster and more robust.

## 2. Mental Model

**A two-stage relay: a spectrogram *artist* hands off to a sound *engineer*.**

```
  text  ──► [ Tacotron 2 ]  ──►  mel spectrogram  ──► [ Vocoder ]  ──►  waveform
 "hello"   encoder+attention      80 × T  "picture"    WaveGlow /        .wav
           +autoregressive        of the voice         HiFi-GAN
           decoder
```

- **Stage 1 (Tacotron 2)** reads the text and *paints* a mel spectrogram — a time × frequency picture of how the voice should sound. It **attends to one part of the text at a time**, sweeping left-to-right like a person reading aloud.
- **Stage 2 (vocoder)** is a different model that turns that picture into actual air-pressure samples. **Tacotron 2 by itself produces no audio.**

The single most important diagnostic is the **attention alignment** — a heatmap of decoder steps (time) vs encoder steps (text). A clean **monotonic diagonal** means the model is reading the text in order. A smeared, looping, or jumping alignment *is* the failure mode you hear as babbling, repeated words, or speech that cuts off early.

## 3. Key Concepts

- **Mel spectrogram** — the intermediate representation both stages share. Typically **80 mel bands**, hop ≈ 256 samples (~11.6 ms) / window 1024 at 22.05 kHz. Log-compressed magnitude; perceptually spaced frequency bins.
- **Encoder** — character embedding → 3 conv layers (model long-range context) → a **bi-directional LSTM**, producing one encoded state per input symbol.
- **Location-sensitive attention** — additive (Bahdanau) attention *augmented with the cumulative previous attention weights*. This location feature nudges attention to move **forward and monotonically**, discouraging skipping/repeating a symbol.
- **Autoregressive decoder** — a 2-layer LSTM that predicts **one mel frame at a time**, conditioning on the previously generated frame through a small **Pre-Net** (2 dense layers with dropout — the dropout is kept *on* at inference and acts as a regularizer against exposure bias).
- **Stop token** — a sigmoid head predicting "utterance finished?" each step, so generation knows when to halt. Without it (or with a miscalibrated one) decoding runs away — hence a hard `max_decoder_steps` cap.
- **Post-Net** — 5 conv layers that predict a *residual* added to the decoder's mel output to sharpen fine detail.
- **Vocoder** — a **separate** model (WaveNet / WaveGlow / HiFi-GAN) mapping mel → waveform. Its mel feature config **must match** Tacotron 2's exactly.
- **Teacher forcing** — trained by feeding *ground-truth* previous frames; at inference the model feeds its *own* predictions, and this train/inference gap (**exposure bias**) is where attention errors accumulate.

## 4. Setup

The conceptual examples below use only **NumPy** and run on any CPU. Real end-to-end synthesis needs **PyTorch** plus a model download (gated behind an env var).

```bash
%pip install numpy torch torchaudio
```

For turnkey weights without wiring the pieces yourself:

- **NVIDIA Tacotron 2 + WaveGlow** via `torch.hub` (used, gated, in Example 3).
- **Coqui TTS** — `pip install TTS`, then `tts --model_name tts_models/en/ljspeech/tacotron2-DDC ...` (bundles a vocoder).
- **SpeechBrain** — `from speechbrain.inference.TTS import Tacotron2` + a HiFi-GAN vocoder. See [[speechbrain]].

In [1]:
import numpy as np

# The conceptual examples use only NumPy. PyTorch is optional and only needed
# for the real-synthesis cell (Example 3), which is gated.
print("numpy:", np.__version__)

try:
    import torch
    print("torch:", torch.__version__, "(real synthesis available)")
except Exception as e:
    torch = None
    print("torch not installed:", type(e).__name__,
          "-> NumPy examples still run; Example 3 prints the call shape only.")

numpy: 2.5.0


torch: 2.12.1 (real synthesis available)


## 5. Worked Examples

### Example 1 — Text → mel frames → duration

Tacotron 2 emits an **80-band mel spectrogram**. The *time* axis is in frames, and frame count maps to audio duration through the vocoder's hop length. This is the arithmetic you reach for constantly when sizing buffers, batching, or sanity-checking output length.

In [2]:
# Example 1 - mel spectrogram geometry and the frame <-> seconds mapping.
SR       = 22_050   # LJSpeech sample rate Tacotron 2 is usually trained on (Hz)
N_MELS   = 80       # mel bands (the spectrogram's frequency resolution)
HOP      = 256      # samples between consecutive frames
WIN      = 1024     # FFT window length

frame_ms = 1000 * HOP / SR
print(f"sample rate = {SR} Hz, hop = {HOP} samples")
print(f"-> one mel frame every {frame_ms:.2f} ms  ({SR/HOP:.1f} frames/sec)\n")

# Rough rule of thumb: English speech ~ 0.06 s/char including spaces & prosody.
for text in ["Hi.", "Hello there, friend.", "The quick brown fox jumps over the lazy dog."]:
    est_secs   = 0.06 * len(text)
    est_frames = int(round(est_secs * SR / HOP))
    mel_shape  = (N_MELS, est_frames)
    print(f"{len(text):>3} chars ~ {est_secs:4.2f}s -> mel {mel_shape}  "
          f"= {N_MELS*est_frames:,} values")

# A real Tacotron 2 mel for a short clip is just a small float matrix:
mel = np.random.randn(N_MELS, 130).astype(np.float32)  # stand-in for 80 x ~1.5s
print(f"\nexample mel: shape={mel.shape}, dtype={mel.dtype}, "
      f"~{mel.nbytes/1024:.1f} KiB")

sample rate = 22050 Hz, hop = 256 samples
-> one mel frame every 11.61 ms  (86.1 frames/sec)

  3 chars ~ 0.18s -> mel (80, 16)  = 1,280 values
 20 chars ~ 1.20s -> mel (80, 103)  = 8,240 values
 44 chars ~ 2.64s -> mel (80, 227)  = 18,160 values

example mel: shape=(80, 130), dtype=float32, ~40.6 KiB


### Example 2 — Location-sensitive attention & the stop token

The decoder produces each mel frame by reading a **context vector**: a weighted sum of encoder states, where the weights are the attention over the input symbols. A healthy run gives a **monotonic diagonal** alignment. Below we build a synthetic alignment, derive context vectors from it, and show the stop-token logic that ends decoding.

In [3]:
# Example 2 - synthesize an attention alignment, read context vectors, decide when to stop.
rng = np.random.default_rng(0)

T_in  = 8    # encoder steps (text symbols)
T_out = 20   # decoder steps (mel frames)
D     = 4    # encoder state dimension

# A clean, monotonic alignment: each decoder step peaks over an advancing input symbol.
centers = np.linspace(0, T_in - 1, T_out)          # diagonal sweep across the text
idx     = np.arange(T_in)
align   = np.exp(-0.5 * ((idx[None, :] - centers[:, None]) / 0.7) ** 2)
align  /= align.sum(axis=1, keepdims=True)          # each decoder row sums to 1

print("attention alignment (decoder steps x text symbols), rounded:")
for t in range(0, T_out, 4):
    row = " ".join(f"{w:.2f}" for w in align[t])
    print(f"  dec step {t:>2}: [{row}]  -> peak at symbol {align[t].argmax()}")

# Monotonic check: the peak symbol must never move backwards.
peaks = align.argmax(axis=1)
print("\npeak symbol per step:", peaks.tolist())
print("monotonic (no backtracking)?", bool(np.all(np.diff(peaks) >= 0)))

# Context vector at each step = attention-weighted sum of encoder states.
encoder_states = rng.standard_normal((T_in, D)).astype(np.float32)
context        = align @ encoder_states             # (T_out, D)
print(f"\ncontext vectors: {context.shape}  (one D={D} vector feeds each decoder step)")

# Stop token: a sigmoid head; decoding halts the first step its prob crosses 0.5.
def sigmoid(x): return 1 / (1 + np.exp(-x))
stop_logits = np.linspace(-6, 4, T_out)             # rises as the utterance completes
stop_prob   = sigmoid(stop_logits)
stop_at = next((t for t, p in enumerate(stop_prob) if p > 0.5), T_out)
print(f"stop-token fires at decoder step {stop_at} of {T_out} "
      f"(prob {stop_prob[stop_at]:.2f}) -> emit {stop_at} mel frames")

attention alignment (decoder steps x text symbols), rounded:
  dec step  0: [0.73 0.26 0.01 0.00 0.00 0.00 0.00 0.00]  -> peak at symbol 0
  dec step  4: [0.06 0.45 0.43 0.05 0.00 0.00 0.00 0.00]  -> peak at symbol 1
  dec step  8: [0.00 0.01 0.23 0.57 0.18 0.01 0.00 0.00]  -> peak at symbol 3
  dec step 12: [0.00 0.00 0.00 0.07 0.48 0.40 0.04 0.00]  -> peak at symbol 4
  dec step 16: [0.00 0.00 0.00 0.00 0.01 0.25 0.57 0.16]  -> peak at symbol 6

peak symbol per step: [0, 0, 1, 1, 1, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 6, 6, 6, 7, 7]
monotonic (no backtracking)? True

context vectors: (20, 4)  (one D=4 vector feeds each decoder step)
stop-token fires at decoder step 12 of 20 (prob 0.58) -> emit 12 mel frames


### Example 3 — Real synthesis with NVIDIA Tacotron 2 + WaveGlow (gated)

This needs PyTorch and a one-time `torch.hub` download (~hundreds of MB), so it runs only when `RUN_TACOTRON=1`. Otherwise it prints the exact call shape. Note how the two stages are explicit: **Tacotron 2 → mel**, then **WaveGlow → audio**.

In [4]:
# Example 3 - end-to-end synthesis (gated behind RUN_TACOTRON + torch + download).
import os

ready = os.getenv("RUN_TACOTRON") and torch is not None

if ready:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    # Stage 1: text -> mel spectrogram
    tacotron2 = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub",
                               "nvidia_tacotron2", model_math="fp32").to(device).eval()
    utils = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub",
                           "nvidia_tts_utils")
    # Stage 2: mel -> waveform
    waveglow = torch.hub.load("NVIDIA/DeepLearningExamples:torchhub",
                              "nvidia_waveglow", model_math="fp32").to(device).eval()
    waveglow = utils.to_fp32_module(waveglow) if hasattr(utils, "to_fp32_module") else waveglow

    text = "Hello world, this is Tacotron 2 speaking."
    seq, lengths = utils.prepare_input_sequence([text])
    with torch.no_grad():
        mel, _, _ = tacotron2.infer(seq.to(device), lengths.to(device))
        audio = waveglow.infer(mel)
    wav = audio[0].cpu().numpy()
    print(f"mel: {tuple(mel.shape)}  audio: {wav.shape}  "
          f"~{wav.shape[-1]/22050:.2f}s @ 22.05 kHz")
    # from scipy.io.wavfile import write; write("out.wav", 22050, wav)
else:
    print("RUN_TACOTRON not set (or torch missing) - showing the call shape:\n")
    print("  tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',")
    print("                             'nvidia_tacotron2').eval()")
    print("  waveglow  = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',")
    print("                             'nvidia_waveglow').eval()")
    print("  utils     = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',")
    print("                             'nvidia_tts_utils')")
    print("  seq, lens = utils.prepare_input_sequence(['Hello world.'])")
    print("  mel, _, _ = tacotron2.infer(seq, lens)   # stage 1: text -> mel")
    print("  audio     = waveglow.infer(mel)          # stage 2: mel -> waveform")

RUN_TACOTRON not set (or torch missing) - showing the call shape:

  tacotron2 = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                             'nvidia_tacotron2').eval()
  waveglow  = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                             'nvidia_waveglow').eval()
  utils     = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub',
                             'nvidia_tts_utils')
  seq, lens = utils.prepare_input_sequence(['Hello world.'])
  mel, _, _ = tacotron2.infer(seq, lens)   # stage 1: text -> mel
  audio     = waveglow.infer(mel)          # stage 2: mel -> waveform


## 6. Gotchas & Pitfalls

- **Attention is the failure point.** On long, repetitive, or out-of-domain text the alignment derails — you hear **repeated words, skipped words, early cutoff, or babbling**. Always plot the alignment when debugging; a clean diagonal is your health check.
- **Mel alone is not audio.** You *must* run a vocoder. And the mel config — `sample_rate`, `n_mels`, `hop`, `win`, `fmin/fmax` — **must match exactly** between Tacotron 2 and the vocoder, or you get noise/garbage even though both models "work".
- **Autoregressive = slow.** Frames are generated one at a time with no parallelism; cost grows linearly with audio length. This rules out tight real-time budgets.
- **Stop-token miscalibration** truncates audio (fires early) or runs away (never fires). Keep a hard `max_decoder_steps` cap as a backstop.
- **Text normalization is on you.** Tacotron 2 expects cleaned text or phonemes; raw `"$5"` or `"Dr."` won't be expanded — run a front-end (e.g. `nemo_text_processing`, `unidecode`, a G2P) first.
- **Single speaker by default.** Multi-speaker / styles need speaker embeddings or a different model.
- **Griffin-Lim is a debug toy, not a vocoder.** It's handy to quickly *hear* whether the mel is sane, but sounds robotic/buzzy — never ship it as the production vocoder.

## 7. When to Use vs Alternatives

| Option | Trade-off vs Tacotron 2 |
|---|---|
| **FastSpeech 2** | Non-autoregressive → much faster and more robust (explicit duration predictor, no attention to derail), but needs alignment/duration supervision and has flatter prosody by default. |
| **[[vits]]** | Fully end-to-end (text → waveform, *no separate vocoder*), excellent quality, faster inference; training is more complex. The modern default for high-quality open TTS. |
| **[[styletts2]] / [[bark]]** | More expressive, style/zero-shot capable, larger and heavier. Reach for these when you need controllable prosody or voice cloning. |
| **[[coqui-tts]]** | A framework that *packages* Tacotron 2 (and others) with vocoders and pretrained weights — easiest way to actually run Tacotron 2 today. |
| **Cloud APIs ([[elevenlabs]], [[azure-speech]], [[amazon-polly]])** | Turnkey, scalable, no GPUs to manage, but pay per character and you don't control the weights. |

**Choose Tacotron 2 when** you want a clear, hackable, single-speaker autoregressive baseline, value naturalness over latency, or are studying the architecture that defined modern TTS. **Choose a non-autoregressive / end-to-end model** when you need speed, robustness on arbitrary text, or production throughput.

## 8. Resources

- **Paper** — *Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrogram Predictions* (Shen et al., 2018): https://arxiv.org/abs/1712.05884
- **NVIDIA reference implementation** (training + inference): https://github.com/NVIDIA/tacotron2
- **`torch.hub` Tacotron 2 + WaveGlow example** (used in Example 3): https://pytorch.org/hub/nvidia_deeplearningexamples_tacotron2/
- **Audio samples** from the paper: https://google.github.io/tacotron/publications/tacotron2/
- **Coqui TTS** — practical pretrained Tacotron 2 + vocoders: https://github.com/coqui-ai/TTS